# 🔭 Analyse multivariée — méthodes factorielles, tests & réduction de dimension

Notebook **Wiki + Tutoriel + Cheat-sheet** sur l'**analyse multivariée** : comment
résumer, cartographier et tester un jeu de données décrit par plusieurs variables.

**Trois blocs** :

1. **Tests statistiques multivariés** — régression (linéaire/logistique), ANOVA, MANOVA.
2. **Analyse factorielle** — PCA, CA, MCA, MFA, FAMD, GPA : réduire l'information à
   quelques axes (« facteurs ») et lire la proximité entre individus / modalités.
3. **Réduction de dimension non-linéaire** — Isomap, LLE, MDS, t-SNE, UMAP.

**Choix des libs (2026)** :

- **`prince` 0.19** est le backbone unique des méthodes factorielles (API scikit-learn,
  couvre PCA/CA/MCA/MFA/FAMD/GPA). ⚠️ Son API a été **entièrement refondue en 2022** —
  tous les vieux tutos (`plot_row_coordinates`, `mapping`, `explained_inertia_`) sont obsolètes.
- **`fanalysis`** (utilisé dans la 1ʳᵉ version de ce notebook) est **abandonné** et
  n'installe plus proprement → ses graphes utiles sont **réimplémentés** en helpers matplotlib.
- **`scikit-learn`** pour la PCA « pipeline ML » et tout le `manifold` ; **`umap-learn`** ajouté.
- **`statsmodels`** pour les tests (p-values, traces de Wilks/Pillai).

**Datasets** (tous programmatiques ou inline — rien à télécharger) : `iris`, `tips`,
`titanic` (catégoriel, pour la MCA), + 2 tables inline (hair×eye, dégustation de vins).

**Renvois** : visualisation EDA → `EDA_Visualisation_Introduction.ipynb` ;
preprocessing (encodage, scalers) → `Structures_Preprocessing.ipynb`.

## 1. Setup, imports et charte graphique

On centralise les imports en tête et on affiche les versions pour la reproductibilité.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prince
import statsmodels.api as sm
import umap
import pacmap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA as SkPCA
from sklearn.decomposition import FactorAnalysis, KernelPCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import MDS, TSNE, Isomap, LocallyLinearEmbedding
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA

print(f"pandas      : {pd.__version__}")
print(f"scikit-learn: {version('scikit-learn')}")
print(f"statsmodels : {version('statsmodels')}")
print(f"prince      : {version('prince')}")
print(f"umap-learn  : {version('umap-learn')}")

**Charte graphique unique** (`CHART`) : 8 couleurs nommées sémantiquement, appliquées
partout via `apply_chart_style()`. Bonne pratique de reporting — voir
`EDA_Visualisation_Introduction.ipynb` pour la justification détaillée.

In [ ]:
CHART: dict[str, str] = {
    "primary_1":   "#00798c",  # Teal     — info / catégorie principale
    "mauvais":     "#d1495b",  # Crimson  — bad / nul / critique
    "moyen":       "#edae49",  # Saffron  — moyen / warning
    "accent":      "#66a182",  # Sage     — accent / bon / highlight
    "accent_dark": "#2e4057",  # Navy     — texte fort, valeur max highlight
    "lavender":    "#9d83b8",  # Violet                — secondaire 1
    "dusty_rose":  "#b8848e",  # Rose terne            — secondaire 2
    "beige":       "#c9b78b",  # Beige                 — neutre / background
}
PALETTE: list[str] = list(CHART.values())


def apply_chart_style() -> None:
    """Applique la charte graphique au runtime matplotlib + seaborn (idempotent)."""
    sns.set_theme(style="whitegrid", palette=PALETTE)
    plt.rcParams.update({
        "figure.figsize": (10, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": CHART["accent_dark"],
        "axes.labelcolor": CHART["accent_dark"],
        "axes.titleweight": "bold",
        "grid.alpha": 0.3,
        "font.size": 11,
    })


apply_chart_style()

Le dataset fil rouge est **iris** (150 fleurs × 4 mesures numériques + l'espèce).
On le charge une fois et on isole la partie numérique.

In [ ]:
iris = sns.load_dataset("iris")
iris_num = iris.drop(columns=["species"])
iris_species = iris["species"]
print("iris:", iris.shape, "| espèces:", list(iris_species.unique()))
iris.head()

## 2. Tests statistiques multivariés

Avant de *réduire*, on *teste* les relations entre variables. Deux familles :

- **Régression** : expliquer/prédire une variable cible à partir des autres.
- **ANOVA / MANOVA** : tester si les moyennes d'une (ANOVA) ou plusieurs (MANOVA)
  variables numériques diffèrent selon les modalités d'un facteur catégoriel.

### 2.1 Régression linéaire

🎯 **Quand & pourquoi.** On veut **expliquer ou prédire une variable numérique
continue** à partir d'autres variables. Ici : prédire le pourboire (`tip`) à partir
de la facture et de la taille de la table. Sert aussi à **quantifier l'effet** de
chaque variable, toutes choses égales par ailleurs.

📊 **Données.** Cible **numérique continue** ; prédicteurs numériques (ou catégoriels
encodés). On travaille sur les colonnes numériques de `tips`.

🧮 **Principe & maths.** On cherche les coefficients $\beta$ qui **minimisent la somme
des carrés des résidus** (moindres carrés ordinaires, OLS) :

$$\hat{y} = \beta_0 + \sum_j \beta_j x_j, \qquad
\hat{\beta} = \arg\min_\beta \sum_i (y_i - \hat{y}_i)^2 = (X^\top X)^{-1} X^\top y$$

Le coefficient $\beta_j$ se lit : *« +1 unité de $x_j$ ⟹ $+\beta_j$ unités de $y$,
les autres variables fixées »*. La qualité globale se mesure par le
$R^2 = 1 - \frac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}$ (part de variance expliquée).

🔍 **Deux backends complémentaires** : **scikit-learn** (prédiction : coefs, R², RMSE)
et **statsmodels** (inférence : p-values, IC à 95 %, $R^2$ ajusté).

In [ ]:
tips = sns.load_dataset("tips")
tips_num = tips.select_dtypes(include="number")
features = tips_num.columns.drop("tip")
X_lin = tips_num[features]
y_lin = tips_num["tip"]
print("X", X_lin.shape, "| y", y_lin.shape)
X_lin.head()

On encapsule l'ajustement sklearn dans une fonction typée réutilisable.

In [ ]:
def fit_linear_sklearn(X: pd.DataFrame, y: pd.Series) -> LinearRegression:
    """Ajuste une régression linéaire OLS (scikit-learn) et affiche R²/RMSE."""
    model = LinearRegression().fit(X, y)
    r2 = model.score(X, y)
    rmse = float(np.sqrt(((y - model.predict(X)) ** 2).mean()))
    print(f"intercept = {model.intercept_:.3f}")
    print(f"coefs     = {dict(zip(X.columns, np.round(model.coef_, 3)))}")
    print(f"R²        = {r2:.3f} | RMSE = {rmse:.3f}")
    return model


reg = fit_linear_sklearn(X_lin, y_lin)

Visualisation : nuage de points sur la variable la plus explicative + droite de
régression (les autres variables fixées à leur moyenne).

In [ ]:
x0 = X_lin.iloc[:, 0]
fig, ax = plt.subplots()
ax.scatter(x0, y_lin, color=CHART["primary_1"], alpha=0.5, label="observations")
xseq = np.linspace(x0.min(), x0.max(), 100)
others_mean = (reg.coef_[1:] * X_lin.iloc[:, 1:].mean().to_numpy()).sum()
ax.plot(xseq, reg.intercept_ + others_mean + reg.coef_[0] * xseq,
        color=CHART["mauvais"], lw=2.5, label="droite de régression")
ax.set(xlabel=X_lin.columns[0], ylabel="tip", title="Régression linéaire (sklearn)")
ax.legend()
plt.show()

La version **statsmodels** ajoute la constante explicitement et donne une table
d'inférence complète (`summary()`) : p-values, IC à 95 %, tests de significativité.

In [ ]:
X_cst = sm.add_constant(X_lin, prepend=False)
ols_res = sm.OLS(y_lin, X_cst).fit()
print("R²_adj =", round(ols_res.rsquared_adj, 3))
print(ols_res.summary())

📖 **Lecture du résultat.**

- **$R^2 = 0{,}47$** : le modèle explique **47 %** de la variabilité du pourboire — correct
  mais loin d'être parfait (le pourboire dépend aussi de l'humeur, du service…).
- **`total_bill`** : coefficient $\approx 0{,}09$, **p-value ≈ 0** ⟹ effet **très
  significatif** : +1 \$ de facture ⟹ +0,09 \$ de pourboire (~9 %, cohérent avec un
  pourcentage d'usage). **`size`** est significatif aussi (p ≈ 0,02) mais plus faible.
- **RMSE ≈ 1,0 \$** : erreur typique de prédiction d'environ 1 dollar.

⚠️ **Hypothèses OLS à vérifier** (sinon l'inférence est biaisée) : relation **linéaire**,
résidus **indépendants**, de **variance constante** (homoscédasticité) et **normaux**.
Les inspecter via un *residual plot* et un *QQ-plot*.

✅ **À retenir** : la régression linéaire **quantifie un effet** et le **teste** ;
sklearn pour prédire, statsmodels pour interpréter/justifier.

### 2.2 Régression logistique

🎯 **Quand & pourquoi.** On veut **prédire/expliquer une variable catégorielle**
(binaire ici : le sexe du payeur). Au lieu d'une valeur continue, on modélise la
**probabilité** d'appartenir à une classe.

📊 **Données.** Cible **catégorielle** (binaire ou multiclasse) ; prédicteurs numériques
ou encodés. Ici `total_bill, tip, size` → `sex`.

🧮 **Principe & maths.** On passe la combinaison linéaire dans la **sigmoïde** pour la
ramener dans $[0,1]$ :

$$P(y=1 \mid x) = \sigma(\beta_0 + \beta^\top x), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

Les coefficients s'interprètent en **log-odds** ; leur exponentielle donne l'**odds-ratio**
$e^{\beta_j}$ : facteur multiplicatif sur la **cote** $\frac{P(1)}{P(0)}$ quand $x_j$
augmente de 1. $e^{\beta_j} > 1$ ⟹ augmente la probabilité, $< 1$ ⟹ la diminue.

🔍 `penalty=None` désactive la régularisation (l'ancien `penalty='none'` — une chaîne —
est supprimé des versions récentes de scikit-learn).

In [ ]:
X_log = tips[["total_bill", "tip", "size"]]
encoder = LabelEncoder()
y_log = encoder.fit_transform(tips["sex"])
print("classes:", list(encoder.classes_))

logit = LogisticRegression(penalty=None, solver="newton-cg", max_iter=200).fit(X_log, y_log)
coef_table = pd.DataFrame(
    np.concatenate([logit.intercept_.reshape(-1, 1), logit.coef_], axis=1),
    index=["coef"],
    columns=["constante"] + list(X_log.columns),
).T
coef_table.round(4)

La version statsmodels (`Logit`) fournit les p-values et le pseudo-$R^2$ de McFadden.

In [ ]:
logit_sm = sm.Logit(y_log, sm.add_constant(X_log.to_numpy())).fit(disp=0)
print("pseudo-R² (McFadden) =", round(logit_sm.prsquared, 3))
print(logit_sm.summary())

Les **odds-ratios** ($e^{\beta}$) rendent les coefficients lisibles : un OR de 1,04 sur
`total_bill` signifie *« +1 \$ de facture multiplie par 1,04 la cote d'être un homme »*.

In [ ]:
odds = pd.DataFrame({
    "coef (log-odds)": logit_sm.params,
    "odds-ratio": np.exp(logit_sm.params),
    "p-value": logit_sm.pvalues,
}, index=["constante"] + list(X_log.columns))
odds.round(3)

📖 **Lecture du résultat.**

- Tous les **odds-ratios ≈ 1** et toutes les **p-values > 0,05** ⟹ aucune variable n'est
  significative. **Pseudo-$R^2$ de McFadden ≈ 0,02** (très faible : un bon ajustement est
  plutôt > 0,2).
- **Conclusion honnête** : facture, pourboire et taille de table **ne permettent pas**
  de prédire le sexe du payeur — et c'est le **bon** enseignement (ne pas sur-interpréter
  un modèle non significatif).

⚠️ **Pièges** : la régression logistique suppose une **séparation linéaire en log-odds** ;
elle est sensible à la **multicolinéarité** (ici `tip` et `total_bill` sont corrélés) et
au **déséquilibre des classes**.

✅ **À retenir** : lire les coefficients en **odds-ratios**, toujours vérifier la
**significativité** avant de conclure à un effet.

### 2.3 ANOVA — une variable numérique vs un facteur

🎯 **Quand & pourquoi.** Comparer la **moyenne d'une variable numérique** entre **plusieurs
groupes** définis par une variable catégorielle. Question : *« la longueur du sépale
diffère-t-elle selon l'espèce ? »* C'est le test de référence pour « 1 numérique × 1 facteur ».

📊 **Données.** Une variable **numérique** (cible) + un **facteur catégoriel** (≥ 2 modalités).

🧮 **Principe & maths.** On **décompose la variance totale** en variance **inter-groupes**
(expliquée par le facteur) et **intra-groupes** (résiduelle). Le test $F$ confronte les deux ;
$H_0$ = « toutes les moyennes de groupe sont égales » :

$$F = \frac{\text{SCE}_{\text{inter}} / (k-1)}{\text{SCE}_{\text{intra}} / (n-k)}, \qquad
\eta^2 = \frac{\text{SCE}_{\text{inter}}}{\text{SCE}_{\text{totale}}} \;(\text{taille d'effet})$$

🔍 Le boxplot ci-dessous **visualise** déjà la séparation : si les boîtes ne se chevauchent
pas, l'ANOVA sera très significative.

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(x="species", y="sepal_length", data=iris,
            hue="species", palette=PALETTE[:3], legend=False, ax=ax)
ax.set_title("Distribution de sepal_length par espèce")
plt.show()

On ajuste un modèle linéaire `sepal_length ~ species` puis on lit la table ANOVA
(type II). La p-value minuscule confirme que l'espèce explique la longueur du sépale.

In [ ]:
lm = ols("sepal_length ~ species", data=iris).fit()
aov = sm.stats.anova_lm(lm, typ=2)
print(aov)

📖 **Lecture du résultat.**

- **$F \approx 119$**, **p-value $\approx 1{,}7\times10^{-31}$** ⟹ on **rejette $H_0$** sans
  ambiguïté : l'espèce explique très fortement la longueur du sépale.
- La taille d'effet $\eta^2 = \frac{63{,}2}{63{,}2 + 39{,}0} \approx 0{,}62$ : **62 %** de la
  variance de `sepal_length` est portée par l'espèce — effet **massif**.

⚠️ **Hypothèses** : résidus **normaux** dans chaque groupe (Shapiro), **homoscédasticité**
(test de Levene) et **indépendance**. Si l'homoscédasticité est violée → ANOVA de **Welch**.

✅ **À retenir** : ANOVA = « les moyennes diffèrent-elles ? » ; toujours accompagner la
p-value d'une **taille d'effet** ($\eta^2$) pour juger de l'**ampleur**, pas que de la significativité.

### 2.4 MANOVA — plusieurs variables numériques vs un facteur

🎯 **Quand & pourquoi.** Comme l'ANOVA, mais avec **plusieurs variables numériques
dépendantes en même temps**. Question : *« les espèces diffèrent-elles sur l'ensemble
{longueur/largeur sépale + pétale} pris conjointement ? »* Utile quand les variables sont
**corrélées** (les tester séparément ignorerait leurs liens et gonflerait le risque d'erreur).

📊 **Données.** **Plusieurs** variables **numériques** (cibles) + un **facteur catégoriel**.

🧮 **Principe & maths.** On teste l'égalité des **vecteurs de moyennes** entre groupes. On
forme les matrices de covariance **inter-groupes** $\mathbf{H}$ et **intra** $\mathbf{E}$,
et on résume $\mathbf{H}\mathbf{E}^{-1}$ par ses valeurs propres. Les statistiques usuelles :

- **Lambda de Wilks** $\Lambda = \prod \frac{1}{1+\lambda_i} \in [0,1]$ — **proche de 0 = forte séparation**.
- **Trace de Pillai** — la plus **robuste** aux écarts d'hypothèses.

🔍 On lit la **p-value** associée (via une approximation en $F$).

In [ ]:
maov = MANOVA.from_formula(
    "sepal_length + sepal_width + petal_length + petal_width ~ species",
    data=iris,
)
mv = maov.mv_test()
wilks = mv.results["species"]["stat"].loc["Wilks' lambda"].astype(float)
print("Wilks' lambda (species):", wilks.round(5).to_dict())

📖 **Lecture du résultat.**

- **Wilks $\Lambda \approx 0{,}023$** (très proche de 0) avec **$F \approx 199$** et
  **p-value $\approx 0$** ⟹ les espèces ont des **profils multivariés radicalement
  différents** sur les 4 mesures prises ensemble.
- C'est précisément ce qui rend iris séparable et fait le succès de la PCA en §3.2.

⚠️ **Hypothèses** : **normalité multivariée** et **égalité des matrices de covariance**
(test de **Box's M**). En cas de doute, préférer la **trace de Pillai**, plus robuste.

✅ **À retenir** : la MANOVA teste les variables **conjointement** ; un $\Lambda$ proche de 0
signale une forte séparation des groupes dans l'espace multivarié.

## 3. Analyse factorielle — réduire et cartographier

L'analyse factorielle cherche **peu d'axes** (« facteurs ») qui résument l'essentiel
de l'information. Le choix de la méthode dépend du **type de données**. L'arbre de
décision ci-dessous (à garder sous le coude) résume la logique :

![Arbre de décision — analyse multivariée](images/arbre_decision_multivarie.png)

**Lecture de l'arbre** :

- **Données catégorielles ?**
  - **Oui + numériques aussi** → **FAMD** (données mixtes).
  - **Oui, que du catégoriel** → **MCA** (>2 variables) ou **CA** (table de contingence 2 variables).
  - **Non (que du numérique)** :
    - **groupes de colonnes** → **MFA**.
    - **analyse de formes** → **GPA**.
    - sinon → **PCA**.

### 3.0 Rappels mathématiques

**L'idée commune.** Un tableau de données est un **nuage de points** dans un espace à
$p$ dimensions (une par variable). Ce nuage a une **forme** (sa dispersion = son
**inertie**). L'analyse factorielle cherche les **directions selon lesquelles le nuage
s'étire le plus** : on les ordonne, on n'en garde que quelques-unes, et on **projette**
dessus pour voir l'essentiel en 2D.

**La mécanique (commune à toutes les méthodes).** On construit une matrice (individus ×
variables, ou table de contingence), on la **standardise/pondère** selon la méthode, puis
on en extrait les **valeurs et vecteurs propres** — concrètement une **SVD** $X = U\Sigma V^\top$ :

- les **vecteurs propres** ($V$) donnent les **axes factoriels** (directions d'étirement) ;
- les **valeurs propres** $\lambda_k = \sigma_k^2$ donnent l'**inertie portée par chaque axe**.

**Vocabulaire et formules à connaître** (pour un individu $i$, un axe $k$) :

- **Inertie** = variance totale du nuage = $\sum_k \lambda_k$. Chaque axe en capte $\lambda_k$.
- **% de variance** de l'axe $k$ : $\dfrac{\lambda_k}{\sum_j \lambda_j}$ — combien l'axe « résume ».
- **Coordonnée factorielle** $F_{ik}$ = projection de l'individu sur l'axe (sa position sur la carte).
- **Contribution** $\text{ctr}_{ik} = \dfrac{m_i\,F_{ik}^2}{\lambda_k}$ : part de l'individu dans
  l'inertie de l'axe — **qui *construit* l'axe** (à lire pour nommer l'axe).
- **cos²** $\cos^2_{ik} = \dfrac{F_{ik}^2}{\sum_j F_{ij}^2}$ : **qualité de représentation**
  (proche de 1 = le point est fidèlement projeté sur cet axe ; proche de 0 = méfiance).

> 💡 **Contribution vs cos²** : la **contribution** dit *qui fait l'axe* (pour l'interpréter) ;
> le **cos²** dit *si un point est bien représenté* (pour savoir si on peut lui faire confiance).

**Choisir la méthode selon les données** (voir l'arbre ci-dessus) :

| Méthode | Type de données | Question répondue | Inertie totale |
|---|---|---|---|
| **PCA** | numériques | axes de variance maximale | $p$ (vars standardisées) |
| **FA** | numériques | facteurs **latents** + bruit | — (modèle proba) |
| **CA** | table de contingence (2 var. cat.) | associations lignes ↔ colonnes | $\phi^2 = \chi^2/n$ |
| **MCA** | ≥ 2 variables catégorielles | proximité des modalités | $\frac{J}{Q}-1$ (gonflée) |
| **FAMD** | mixte (num + cat) | PCA ⊕ MCA | mixte |
| **MFA** | numériques en **groupes** | équilibre entre groupes | groupes pondérés |
| **GPA** | configurations de **formes** | alignement (rotation/échelle/translation) | distance de Procruste |

### 3.1 Helpers de visualisation factorielle

Quatre helpers matplotlib typés (ils **remplacent les graphes de `fanalysis`**) :
`scree_plot` (éboulis), `plot_factor_map` (carte des individus), `correlation_circle`
(cercle des corrélations) et `plot_contributions` (barres de contribution). Réutilisables
tels quels dans un autre projet.

In [ ]:
def scree_plot(pov: np.ndarray, title: str = "Éboulis des valeurs propres",
               ax: plt.Axes | None = None) -> plt.Axes:
    """Diagramme d'éboulis : % de variance/inertie expliquée par axe + cumul."""
    if ax is None:
        _, ax = plt.subplots()
    n = len(pov)
    axes_lbl = [f"Axe {i + 1}" for i in range(n)]
    ax.bar(axes_lbl, pov, color=CHART["primary_1"], label="% par axe")
    ax2 = ax.twinx()
    ax2.plot(axes_lbl, np.cumsum(pov), color=CHART["mauvais"], marker="o", lw=2)
    ax2.set_ylim(0, 105)
    ax.set(ylabel="% variance", title=title)
    ax2.set_ylabel("% cumulé")
    return ax


def plot_factor_map(coords: pd.DataFrame, labels: pd.Series | None = None,
                    x: int = 0, y: int = 1, title: str = "Carte factorielle",
                    ax: plt.Axes | None = None) -> plt.Axes:
    """Projette des individus sur 2 axes factoriels, colorés par un label optionnel."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    if labels is None:
        ax.scatter(coords.iloc[:, x], coords.iloc[:, y],
                   color=CHART["primary_1"], s=40, alpha=0.7)
    else:
        for i, lvl in enumerate(pd.unique(labels)):
            mask = np.asarray(labels) == lvl
            ax.scatter(coords.iloc[:, x][mask], coords.iloc[:, y][mask],
                       color=PALETTE[i % len(PALETTE)], s=40, alpha=0.7, label=str(lvl))
        ax.legend(title=getattr(labels, "name", None))
    ax.axhline(0, color="grey", lw=0.6)
    ax.axvline(0, color="grey", lw=0.6)
    ax.set(xlabel=f"Axe {x + 1}", ylabel=f"Axe {y + 1}", title=title)
    return ax


def correlation_circle(corr: pd.DataFrame, x: int = 0, y: int = 1,
                       ax: plt.Axes | None = None) -> plt.Axes:
    """Cercle des corrélations : flèches variables → axes (PCA/FAMD)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color=CHART["accent_dark"], lw=1))
    for var in corr.index:
        cx, cy = corr.iloc[:, x][var], corr.iloc[:, y][var]
        ax.arrow(0, 0, cx, cy, head_width=0.03, color=CHART["primary_1"])
        ax.text(cx * 1.1, cy * 1.1, var, ha="center", color=CHART["accent_dark"])
    ax.axhline(0, color="grey", lw=0.6)
    ax.axvline(0, color="grey", lw=0.6)
    ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), xlabel=f"Axe {x + 1}",
           ylabel=f"Axe {y + 1}", title="Cercle des corrélations")
    ax.set_aspect("equal")
    return ax


def plot_contributions(contrib: pd.DataFrame, axis: int = 0, top: int = 10,
                       ax: plt.Axes | None = None) -> plt.Axes:
    """Barres horizontales des contributions des variables à un axe."""
    if ax is None:
        _, ax = plt.subplots()
    s = contrib.iloc[:, axis].sort_values(ascending=True).tail(top)
    ax.barh(s.index.astype(str), s.to_numpy(), color=CHART["primary_1"])
    ax.set(xlabel="contribution", title=f"Contributions — Axe {axis + 1}")
    return ax

### 3.2 ACP / PCA — variables numériques

La **PCA** projette des variables numériques sur des axes orthogonaux de variance
décroissante. On **centre-réduit** d'abord (sinon les variables à grande échelle
dominent). Deux vues : scikit-learn (pipeline ML) puis prince (API factorielle).

#### 3.2.1 Avec scikit-learn

Deux fonctions typées : `pca_sklearn` (standardisation + ajustement) et
`variance_table` (tableau valeur propre / % variance / % cumulé).

In [ ]:
def pca_sklearn(df: pd.DataFrame, n_components: int) -> tuple[SkPCA, np.ndarray, list[str]]:
    """Standardise les variables numériques et applique une PCA scikit-learn.

    Retourne (modèle ajusté, données standardisées, noms des variables).
    """
    num = df.select_dtypes(include="number")
    cols = list(num.columns)
    X = StandardScaler().fit_transform(num.to_numpy())
    model = SkPCA(n_components=n_components).fit(X)
    return model, X, cols


def variance_table(model: SkPCA) -> pd.DataFrame:
    """Tableau valeur propre / % variance / % cumulé par dimension."""
    n = len(model.explained_variance_ratio_)
    return pd.DataFrame({
        "Dim": [f"Dim {i + 1}" for i in range(n)],
        "Val propre": np.round(model.explained_variance_, 3),
        "% var": np.round(model.explained_variance_ratio_ * 100, 2),
        "% cumulé": np.round(np.cumsum(model.explained_variance_ratio_ * 100), 2),
    })


sk_pca, X_std, var_names = pca_sklearn(iris, n_components=3)
variance_table(sk_pca)

Projection des 150 fleurs sur les 2 premiers axes, colorées par espèce : `setosa`
se détache nettement (axe 1), `versicolor`/`virginica` se chevauchent partiellement.

In [ ]:
scores = pd.DataFrame(sk_pca.transform(X_std),
                      columns=[f"Dim {i + 1}" for i in range(sk_pca.n_components_)])
fig, ax = plt.subplots(figsize=(7, 6))
plot_factor_map(scores, iris_species, title="PCA iris (sklearn) — individus", ax=ax)
plt.show()

#### 3.2.2 Avec prince (API 2026)

`prince.PCA` suit l'API scikit-learn (`.fit`) et expose directement
`eigenvalues_summary`, `row_coordinates`, `column_correlations` (corrélations
variables↔axes, pour le cercle) et `column_contributions_`.

In [ ]:
prince_pca = prince.PCA(n_components=3, random_state=42).fit(iris_num)
prince_pca.eigenvalues_summary

Le **cercle des corrélations** lit l'influence de chaque variable : `petal_length`
et `petal_width` portent l'axe 1, `sepal_width` l'axe 2.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
correlation_circle(prince_pca.column_correlations, ax=ax)
plt.show()

Éboulis (à gauche) et contributions des variables à l'axe 1 (à droite).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
scree_plot(prince_pca.percentage_of_variance_, ax=a1)
plot_contributions(prince_pca.column_contributions_, axis=0, ax=a2)
plt.show()

#### 3.2.3 Combien d'axes retenir ?

Trois critères usuels pour choisir le nombre de composantes :

- **Kaiser** : garder les axes de **valeur propre > 1** (sur données standardisées ;
  un axe doit porter plus qu'une variable d'origine). Souvent **trop sévère**.
- **Coude (scree)** : couper là où l'éboulis « décroche » (cf. graphe ci-dessus).
- **Seuil de variance cumulée** : garder assez d'axes pour atteindre p. ex. **80 %**.

In [ ]:
eigvals = prince_pca.eigenvalues_
cumpov = np.cumsum(prince_pca.percentage_of_variance_)
n_kaiser = int((np.asarray(eigvals) > 1).sum())
n_seuil80 = int(np.argmax(cumpov >= 80) + 1)
print(f"Kaiser (valeur propre > 1) : {n_kaiser} axe(s)")
print(f"Seuil 80% variance cumulée : {n_seuil80} axe(s)")
print(f"Variance cumulée : {np.round(cumpov, 1)}")

#### 3.2.4 cos² — qualité de représentation

Le **cos²** d'un individu sur un axe mesure **à quel point il est bien projeté** :
proche de 1, le point est fidèlement représenté ; proche de 0, il « vit » sur d'autres
axes et son interprétation sur ce plan est trompeuse. On colore le nuage par la qualité
sur le plan 1-2 (somme des cos² des deux axes).

In [ ]:
pc_coords = prince_pca.row_coordinates(iris_num)
cos2 = prince_pca.row_cosine_similarities(iris_num)
quality = cos2[0] + cos2[1]  # qualité sur le plan factoriel 1-2
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(pc_coords[0], pc_coords[1], c=quality, cmap="viridis", s=40)
fig.colorbar(sc, ax=ax, label="cos² (qualité plan 1-2)")
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set(title="PCA iris — individus colorés par qualité (cos²)", xlabel="Axe 1", ylabel="Axe 2")
plt.show()

#### 3.2.5 Biplot

Le **biplot** superpose sur un même plan les **individus** (points) et les **variables**
(flèches = loadings). On lit d'un coup d'œil quelles variables « tirent » quels individus :
ici `petal_length`/`petal_width` pointent vers `virginica`, `sepal_width` vers le haut.

In [ ]:
def biplot(scores: pd.DataFrame, loadings: np.ndarray, var_names: list[str],
           labels: pd.Series | None = None, ax: plt.Axes | None = None) -> plt.Axes:
    """Biplot PCA : individus (points) + variables (flèches) sur le même plan."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    plot_factor_map(scores, labels, title="Biplot PCA", ax=ax)
    scale = np.abs(scores.iloc[:, :2].to_numpy()).max() / np.abs(loadings).max()
    for i, name in enumerate(var_names):
        ax.arrow(0, 0, loadings[i, 0] * scale, loadings[i, 1] * scale,
                 color=CHART["accent_dark"], head_width=0.08, alpha=0.8)
        ax.text(loadings[i, 0] * scale * 1.1, loadings[i, 1] * scale * 1.1,
                name, color=CHART["accent_dark"], weight="bold")
    return ax


loadings = sk_pca.components_[:2].T  # (n_vars, 2)
fig, ax = plt.subplots(figsize=(7, 7))
biplot(scores, loadings, var_names, iris_species, ax=ax)
plt.show()

#### 3.2.6 Variables et individus supplémentaires

Un atout des méthodes factorielles : projeter des **individus supplémentaires** (ou des
variables) **a posteriori**, sans qu'ils n'influencent la construction des axes. Utile
pour positionner de nouvelles observations dans un repère figé. Ici on ajuste la PCA sur
135 fleurs et on **projette** les 15 restantes via `row_coordinates`.

In [ ]:
rng_sup = np.random.RandomState(1)
sup_idx = rng_sup.choice(iris_num.index, size=15, replace=False)
train_idx = iris_num.index.difference(sup_idx)
pca_train = prince.PCA(n_components=2, random_state=42).fit(iris_num.loc[train_idx])
coords_train = pca_train.row_coordinates(iris_num.loc[train_idx])
coords_sup = pca_train.row_coordinates(iris_num.loc[sup_idx])  # projection sans réajuster
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(coords_train[0], coords_train[1], color=CHART["beige"], s=30, label="actifs")
ax.scatter(coords_sup[0], coords_sup[1], color=CHART["mauvais"], s=70,
           marker="*", label="supplémentaires")
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set(title="PCA — individus supplémentaires (projetés a posteriori)",
       xlabel="Axe 1", ylabel="Axe 2")
ax.legend()
plt.show()

#### 3.2.7 Clustering sur les composantes

Workflow classique (esprit **HCPC** de FactoMineR) : **réduire** d'abord avec une PCA,
**puis regrouper** sur les premières composantes (débruitées). On compare le KMeans aux
espèces réelles via l'**ARI** (Adjusted Rand Index, 1 = identique). Pour le clustering
approfondi (silhouette, dendrogrammes, DBSCAN…), voir le notebook dédié.

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(pc_coords[[0, 1]])
ari = adjusted_rand_score(iris_species, km.labels_)
print(f"KMeans(3) sur axes 1-2 — ARI vs espèces réelles : {ari:.3f}")
fig, ax = plt.subplots(figsize=(7, 6))
plot_factor_map(pc_coords, pd.Series(km.labels_, name="cluster"),
                title=f"KMeans sur composantes PCA (ARI={ari:.2f})", ax=ax)
plt.show()

#### 3.2.8 PCA vs Factor Analysis (FA)

On confond souvent PCA et **analyse factorielle (FA)**, mais ce sont deux modèles
distincts :

- **PCA** — purement géométrique : décompose la **variance totale** en axes orthogonaux.
  Pas de modèle probabiliste.
- **FA** — modèle **à variables latentes** : suppose que les variables observées sont
  générées par quelques **facteurs cachés** + un **bruit spécifique** à chaque variable.
  $x = \mathbf{L}f + \varepsilon$. Idéale quand on cherche à *interpréter* des facteurs
  sous-jacents (psychométrie, questionnaires). La rotation **varimax** rend les loadings
  plus lisibles (chaque variable « charge » surtout un facteur).

In [ ]:
X_std_fa = StandardScaler().fit_transform(iris_num.to_numpy())
fa = FactorAnalysis(n_components=2, rotation="varimax", random_state=42).fit(X_std_fa)
fa_loadings = pd.DataFrame(fa.components_.T, index=var_names, columns=["Facteur 1", "Facteur 2"])
print(fa_loadings.round(3))
fa_scores = fa.transform(X_std_fa)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
for i, lvl in enumerate(iris_species.unique()):
    m = pd.Categorical(iris_species).codes == i
    a1.scatter(scores.iloc[:, 0][m], scores.iloc[:, 1][m], color=PALETTE[i], s=25, label=lvl)
    a2.scatter(fa_scores[m, 0], fa_scores[m, 1], color=PALETTE[i], s=25, label=lvl)
a1.set_title("PCA (variance)")
a2.set_title("Factor Analysis (facteurs latents + bruit)")
a1.legend(title="espèce")
plt.show()

### 3.3 AFC / CA — table de contingence (2 variables catégorielles)

L'**analyse des correspondances** décrit l'association entre les lignes et les
colonnes d'une **table de contingence**. Elle décompose l'inertie du $\chi^2$ et
place lignes et colonnes sur un même plan : deux modalités proches sont associées.
Exemple classique : couleur des **yeux** × couleur des **cheveux**.

In [ ]:
hair_eye = pd.DataFrame(
    data=[[326, 38, 241, 110, 3],
          [688, 116, 584, 188, 4],
          [343, 84, 909, 412, 26],
          [98, 48, 403, 681, 85]],
    columns=["Fair", "Red", "Medium", "Dark", "Black"],
    index=["Blue", "Light", "Medium", "Dark"],
)
hair_eye.columns.name = "Cheveux"
hair_eye.index.name = "Yeux"
hair_eye

`prince.CA` donne l'inertie par axe et les coordonnées des lignes (`row_coordinates`)
et colonnes (`column_coordinates`).

In [ ]:
ca = prince.CA(n_components=2, random_state=42).fit(hair_eye)
ca.eigenvalues_summary

Carte simultanée : yeux (teal) et cheveux (rouge). On lit le gradient clair→foncé
le long de l'axe 1 (yeux clairs/cheveux blonds à gauche, foncés à droite).

In [ ]:
row_c = ca.row_coordinates(hair_eye)
col_c = ca.column_coordinates(hair_eye)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(row_c.iloc[:, 0], row_c.iloc[:, 1], color=CHART["primary_1"], s=60)
for lbl, (px, py) in zip(row_c.index, row_c.to_numpy()):
    ax.text(px, py, str(lbl), color=CHART["primary_1"], weight="bold")
ax.scatter(col_c.iloc[:, 0], col_c.iloc[:, 1], color=CHART["mauvais"], s=60, marker="^")
for lbl, (px, py) in zip(col_c.index, col_c.to_numpy()):
    ax.text(px, py, str(lbl), color=CHART["mauvais"], weight="bold")
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set(title="CA — yeux (teal) × cheveux (rouge)", xlabel="Axe 1", ylabel="Axe 2")
plt.show()

### 3.4 ACM / MCA — plusieurs variables catégorielles

La **MCA** étend la CA à **plus de deux variables catégorielles** : c'est une approche
« datamining » qui cartographie les **modalités** pour repérer celles qui vont ensemble.
La 1ʳᵉ version de ce notebook lisait un fichier depuis Google Drive (non reproductible) ;
on le remplace par un sous-ensemble **catégoriel du Titanic** (chargé programmatiquement).

In [ ]:
titanic = sns.load_dataset("titanic")
cat_cols = ["sex", "class", "embark_town", "who", "alive"]
titanic_cat = titanic[cat_cols].dropna().astype(str)
print("Titanic catégoriel:", titanic_cat.shape)
titanic_cat.head()

`prince.MCA` encode automatiquement les variables en one-hot puis applique l'analyse.

In [ ]:
mca = prince.MCA(n_components=2, random_state=42).fit(titanic_cat)
mca.eigenvalues_summary

Carte des **modalités** : on retrouve l'opposition survie (`alive=yes` / `no`) corrélée
au sexe et à la classe — la signature bien connue du naufrage.

In [ ]:
mca_cols = mca.column_coordinates(titanic_cat)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(mca_cols.iloc[:, 0], mca_cols.iloc[:, 1], color=CHART["mauvais"], s=50)
for lbl, (px, py) in zip(mca_cols.index, mca_cols.to_numpy()):
    ax.text(px, py, str(lbl), fontsize=8, color=CHART["accent_dark"])
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set(title="MCA Titanic — carte des modalités", xlabel="Axe 1", ylabel="Axe 2")
plt.show()

### 3.5 AFM / MFA — groupes de variables

L'**analyse factorielle multiple** traite des variables organisées en **groupes**
(ici : 3 experts notent 6 vins sur plusieurs descripteurs). Chaque groupe est
normalisé par sa 1ʳᵉ valeur propre pour qu'**aucun groupe ne domine**, puis on fait
une PCA globale. On peut lire la contribution de chaque groupe aux axes.

In [ ]:
wine = pd.DataFrame(
    data=[[1, 6, 7, 2, 5, 7, 6, 3, 6, 7],
          [5, 3, 2, 4, 4, 4, 2, 4, 4, 3],
          [6, 1, 1, 5, 2, 1, 1, 7, 1, 1],
          [7, 1, 2, 7, 2, 1, 2, 2, 2, 2],
          [2, 5, 4, 3, 5, 6, 5, 2, 6, 6],
          [3, 4, 4, 3, 5, 4, 5, 1, 7, 5]],
    columns=["E1 fruity", "E1 woody", "E1 coffee",
             "E2 red fruit", "E2 roasted", "E2 vanillin", "E2 woody",
             "E3 fruity", "E3 butter", "E3 woody"],
    index=[f"Wine {i + 1}" for i in range(6)],
)
groups = {
    f"Expert {no + 1}": [c for c in wine.columns if c.startswith(f"E{no + 1}")]
    for no in range(3)
}
groups

Les groupes sont passés à **`.fit(X, groups=...)`** (et non plus au constructeur,
comme dans l'ancienne API). `group_contributions_` donne le poids de chaque expert.

In [ ]:
mfa = prince.MFA(n_components=2, random_state=42).fit(wine, groups=groups)
print(mfa.eigenvalues_summary)
mfa.group_contributions_.round(3)

Carte des individus (vins) sur les 2 premiers axes globaux.

In [ ]:
mfa_rows = mfa.row_coordinates(wine)
fig, ax = plt.subplots(figsize=(7, 6))
plot_factor_map(mfa_rows, pd.Series(wine.index, name="Vin"),
                title="MFA — individus (vins)", ax=ax)
plt.show()

### 3.6 AFDM / FAMD — données mixtes (numérique + catégoriel)

La **FAMD** gère un tableau qui mélange variables **numériques et catégorielles** :
mathématiquement, **FAMD = PCA (sur le numérique) ⊕ MCA (sur le catégoriel)**, avec
une pondération qui met les deux types sur un pied d'égalité.

⚠️ Piège pratique : `prince.FAMD` ne reconnaît comme numériques que les colonnes de
dtype **`float`** — on caste donc explicitement les colonnes numériques.

In [ ]:
wine_mixed = pd.DataFrame(
    data=[["A", "A", "A", 2, 5, 7, 6, 3, 6, 7],
          ["A", "A", "A", 4, 4, 4, 2, 4, 4, 3],
          ["B", "A", "B", 5, 2, 1, 1, 7, 1, 1],
          ["B", "A", "B", 7, 2, 1, 2, 2, 2, 2],
          ["B", "B", "B", 3, 5, 6, 5, 2, 6, 6],
          ["B", "B", "A", 3, 5, 4, 5, 1, 7, 5]],
    columns=["c1", "c2", "c3", "n1", "n2", "n3", "n4", "n5", "n6", "n7"],
    index=[f"Wine {i + 1}" for i in range(6)],
)
num_cols = ["n1", "n2", "n3", "n4", "n5", "n6", "n7"]
wine_mixed[num_cols] = wine_mixed[num_cols].astype(float)
oak = pd.Series([1, 2, 2, 2, 1, 1], index=wine_mixed.index, name="Oak type")
wine_mixed

Ajustement et inertie expliquée. La carte des individus colorée par `Oak type`
montre une séparation nette des deux types de chêne sur l'axe 1.

In [ ]:
famd = prince.FAMD(n_components=2, random_state=42).fit(wine_mixed)
print(famd.eigenvalues_summary)
famd_rows = famd.row_coordinates(wine_mixed)
fig, ax = plt.subplots(figsize=(7, 6))
plot_factor_map(famd_rows, oak.astype(str), title="FAMD — individus (Oak type)", ax=ax)
plt.show()

### 3.7 GPA — analyse procustéenne généralisée (formes)

La **GPA** aligne plusieurs **configurations de points** (formes) en supprimant les
différences de **translation, rotation et échelle**, pour ne garder que les
différences de *forme* réelles. Usages : morphométrie, comparaison de capteurs,
consensus de notations spatiales.

La 1ʳᵉ version utilisait une implémentation maison incorrecte (rotation 1-D bricolée
qui mutait le DataFrame en place) ; on la remplace par **`prince.GPA`**. L'entrée est
un tableau 3D `(n_formes, n_points, n_dimensions)`.

In [ ]:
rng = np.random.RandomState(0)
base_shape = np.array([[0, 0], [1, 0], [1.2, 1], [0.4, 1.4], [-0.3, 0.8]], dtype=float)


def _rot(theta: float) -> np.ndarray:
    """Matrice de rotation 2D d'angle theta (radians)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


# 3 copies de la même forme : translatées, tournées, redimensionnées + bruit.
shapes = np.stack([
    base_shape @ _rot(0.0).T + np.array([2.0, 2.0]) + rng.normal(0, 0.03, base_shape.shape),
    base_shape * 1.4 @ _rot(0.7).T + np.array([-1.0, 1.0]) + rng.normal(0, 0.03, base_shape.shape),
    base_shape * 0.7 @ _rot(-0.5).T + np.array([0.5, -1.5]) + rng.normal(0, 0.03, base_shape.shape),
])
print("formes (n_formes, n_points, n_dims):", shapes.shape)

`fit_transform` renvoie les formes **alignées** : avant, 3 pentagones éparpillés ;
après, ils se superposent presque parfaitement (seul le bruit subsiste).

In [ ]:
gpa = prince.GPA()
aligned = np.asarray(gpa.fit_transform(shapes))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
for i in range(shapes.shape[0]):
    a1.plot(*np.vstack([shapes[i], shapes[i][0]]).T, marker="o",
            color=PALETTE[i], label=f"forme {i + 1}")
    a2.plot(*np.vstack([aligned[i], aligned[i][0]]).T, marker="o",
            color=PALETTE[i], label=f"forme {i + 1}")
a1.set_title("Avant alignement")
a2.set_title("Après GPA")
for a in (a1, a2):
    a.legend()
    a.set_aspect("equal")
plt.show()

### 3.8 Kernel PCA — PCA non-linéaire

La **Kernel PCA** applique le *kernel trick* : elle fait une PCA dans un espace de
dimension supérieure (implicite) pour capturer des structures **non-linéaires** que la
PCA classique manque. C'est le pont naturel vers la section 4 (manifold learning).
Le noyau RBF est piloté par `gamma` (à régler).

In [ ]:
X_iris_std = StandardScaler().fit_transform(iris_num.to_numpy())
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=0.5)
kpca_emb = kpca.fit_transform(X_iris_std)
lin_emb = SkPCA(n_components=2).fit_transform(X_iris_std)
codes = pd.Categorical(iris_species).codes
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
for i, lvl in enumerate(iris_species.unique()):
    m = codes == i
    a1.scatter(lin_emb[m, 0], lin_emb[m, 1], color=PALETTE[i], s=25, label=lvl)
    a2.scatter(kpca_emb[m, 0], kpca_emb[m, 1], color=PALETTE[i], s=25, label=lvl)
a1.set_title("PCA linéaire")
a2.set_title("Kernel PCA (RBF)")
a1.legend(title="espèce")
plt.show()

## 4. Réduction de dimension (manifold learning)

Au-delà de la PCA (**linéaire**), les méthodes de **manifold learning** capturent des
structures **non-linéaires**. On les compare toutes sur iris via un helper unifié.

In [ ]:
X_iris = iris_num.to_numpy()


def reduce(method: str, X: np.ndarray) -> np.ndarray:
    """Réduit X à 2D selon la méthode demandée (sklearn manifold + umap + pacmap)."""
    if method == "PaCMAP":
        return np.asarray(pacmap.PaCMAP(n_components=2, random_state=42).fit_transform(X, init="pca"))
    reducers = {
        "PCA": SkPCA(n_components=2),
        "Isomap": Isomap(n_neighbors=10, n_components=2),
        "LLE": LocallyLinearEmbedding(n_neighbors=10, n_components=2),
        "MDS": MDS(n_components=2, normalized_stress="auto", random_state=42),
        "t-SNE": TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42),
        "UMAP": umap.UMAP(n_components=2, random_state=42),
    }
    return reducers[method].fit_transform(X)

**Intuition de chaque méthode** :

- **PCA** — projection **linéaire** de variance maximale (rapide, interprétable).
- **Isomap** — préserve les **distances géodésiques** le long de la variété.
- **LLE** — reconstruit chaque point depuis ses **voisins** (relations locales).
- **MDS** — préserve au mieux les **distances deux-à-deux** (minimise le *stress*).
- **t-SNE** — préserve les **voisinages** locaux (divergence KL) ; superbe pour visualiser des clusters.
- **UMAP** — graphe flou de voisinage ; **rapide**, préserve mieux la structure globale que t-SNE. Standard 2026.
- **PaCMAP** — méthode 2026 qui **équilibre structures locale et globale** via 3 types de paires (voisins / mi-distance / lointains) ; souvent plus fidèle que t-SNE/UMAP et peu sensible à l'init.

In [ ]:
methods = ["PCA", "Isomap", "LLE", "MDS", "t-SNE", "UMAP", "PaCMAP"]
embeddings = {m: reduce(m, X_iris) for m in methods}
{m: emb.shape for m, emb in embeddings.items()}

Grille comparative : `setosa` (teal) toujours isolée ; t-SNE, UMAP et PaCMAP séparent
nettement les trois espèces, là où les méthodes linéaires laissent
`versicolor`/`virginica` se chevaucher.

In [ ]:
species_codes = pd.Categorical(iris_species).codes
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, m in zip(axes.ravel(), methods):
    emb = embeddings[m]
    for i, lvl in enumerate(iris_species.unique()):
        mask = species_codes == i
        ax.scatter(emb[mask, 0], emb[mask, 1], color=PALETTE[i], s=20, alpha=0.7, label=lvl)
    ax.set_title(m)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.ravel()[len(methods):]:
    ax.set_visible(False)
axes.ravel()[0].legend(title="espèce", fontsize=8)
fig.suptitle("Réduction de dimension d'iris — 7 méthodes", weight="bold")
plt.show()

### 4.1 Pièges de t-SNE / UMAP

Ces méthodes sont superbes pour **visualiser** des clusters, mais traîtres si on
sur-interprète :

- **Les distances n'ont pas de sens global** : deux clusters éloignés sur le plot ne
  sont pas forcément « plus différents » que deux proches.
- **La taille et la densité des clusters sont arbitraires** (t-SNE les égalise).
- **Forte sensibilité aux hyperparamètres** : `perplexity` (t-SNE) / `n_neighbors` (UMAP)
  changent radicalement la figure — voir ci-dessous.
- **Stochastiques** : fixer `random_state` pour la reproductibilité.

Règle : utiliser t-SNE/UMAP pour *explorer*, jamais comme seule preuve d'une structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, perp in zip(axes, [5, 30, 50]):
    emb = TSNE(n_components=2, perplexity=perp, max_iter=1000,
               random_state=42).fit_transform(X_iris)
    for i, lvl in enumerate(iris_species.unique()):
        m = codes == i
        ax.scatter(emb[m, 0], emb[m, 1], color=PALETTE[i], s=20, label=lvl)
    ax.set_title(f"perplexity = {perp}")
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].legend(title="espèce", fontsize=8)
fig.suptitle("t-SNE — la forme dépend fortement de perplexity", weight="bold")
plt.show()

## 5. Récapitulatif — quelle méthode choisir

| Situation | Méthode | Section |
|---|---|---|
| Expliquer/prédire une variable numérique | Régression linéaire | 2.1 |
| Expliquer/prédire une variable catégorielle | Régression logistique | 2.2 |
| Comparer des moyennes (1 var.) entre groupes | ANOVA | 2.3 |
| Comparer des moyennes (≥2 var.) entre groupes | MANOVA | 2.4 |
| Résumer des variables **numériques** | **PCA** | 3.2 |
| Interpréter des **facteurs latents** (+ bruit) | **Factor Analysis** | 3.2.8 |
| Associer 2 variables **catégorielles** (contingence) | **CA** | 3.3 |
| Cartographier **≥2 variables catégorielles** | **MCA** | 3.4 |
| Variables numériques en **groupes** | **MFA** | 3.5 |
| Données **mixtes** (num + cat) | **FAMD** | 3.6 |
| Aligner des **formes** | **GPA** | 3.7 |
| Capturer du **non-linéaire** en gardant l'esprit PCA | **Kernel PCA** | 3.8 |
| Visualiser des clusters **non-linéaires** | t-SNE / UMAP / PaCMAP | 4 |
| **Regrouper** après réduction | PCA + KMeans (HCPC) | 3.2.7 |

**Bonnes pratiques transverses** : choisir le nombre d'axes (Kaiser / coude / 80 %, §3.2.3),
vérifier la qualité de projection (**cos²**, §3.2.4), projeter des **individus
supplémentaires** sans réajuster (§3.2.6), et ne jamais sur-interpréter t-SNE/UMAP (§4.1).
Garder l'**arbre de décision** de la section 3 comme aide-mémoire principal.